In [ ]:
from fastai.vision.all import *
from torch.nn import *


class Dat:
    a = torch.from_numpy(np.memmap(f"../tmp/news2016zhMin100.H", "h"))
    m = np.array([open("../tmp/news2016zhMin100.txt").read()]).view("U1")

    def __init__(s, w, b, n=1):
        s.a = s.a.as_strided([len(s.a) - w + 1, w], [1, 1])
        s.b = b
        s.n = n

    def __len__(s):
        return len(s.a) // (s.a.size(1) * s.b * s.n)

    def __iter__(s):
        for x in range(len(s)):
            x = s.a[torch.randint(len(s.a), [s.b])].to("cuda", torch.long)
            yield x, x


class Transpose(Module):
    @staticmethod
    def forward(x):
        return x.transpose(1, 2)


class ConvBn(Sequential):
    def __init__(_, i, w=3, s=1):
        super().__init__(Conv1d(i, i, w, s, w // 2, bias=0), BatchNorm1d(i, track_running_stats=0), Swish())


class ConvTransBn(Sequential):
    def __init__(_, i, w=3, s=1):
        super().__init__(ConvTranspose1d(i, i, w, s, w // 2, s // 2, bias=0), BatchNorm1d(i, track_running_stats=0), Swish())


class Res(Sequential):
    def forward(_, x):
        return super().forward(x) + x


class Cnn(Sequential):
    def __init__(s, *a, n=0):
        super().__init__(*a)
        s.n = n

    def forward(s, x):
        for _ in range(s.n or int(math.log2(x.size(2)))):
            x = super().forward(x)
        return x

In [ ]:
m = Learner(
    DataLoaders(Dat(64, 2 ** 9 * 5, 100), Dat(64, 2 ** 9 * 5, 10000)),
    Sequential(
        Embedding(684, 256),
        Transpose(),
        Cnn(Res(ConvBn(256), ConvBn(256)), AvgPool1d(2)),
        ConvBn(256, 1),
        Conv1d(256, 256, 1, bias=0),
        Cnn(Res(ConvTransBn(256), ConvTransBn(256)), ConvTranspose1d(256, 256, 3, 2, 1, 1, bias=0), n=6),
        ConvBn(256, 1),
        Conv1d(256, 684, 1),
    ).cuda(),
    CrossEntropyLoss(),
    cbs=[
        ShowGraphCallback,
        SaveModelCallback(fname="model5-autoencoder-Copy14", with_opt=1, reset_on_fit=0),
    ],
    metrics=partial(accuracy, axis=1),
    wd=0.01,
)
m.load('model5-autoencoder-Copy14')
m.validate()
# m.lr_find()

In [ ]:
m.fine_tune(100)